# EA2 – Exploración descriptiva del dataset de precios de arriendo en Colombia

Estudiante: Duván Oswaldo Ramírez Duarte

**Proyecto integrador: Ideación, diseño y documentación de la necesidad**

Este notebook realiza la etapa técnica inicial solicitada: obtiene el dataset desde Hugging Face, lo carga en un DataFrame de pandas, presenta una descripción básica y genera automáticamente un resumen en `.txt`.

**Dataset:** `aptologia/precios-arriendo-colombia`  
**Configuración:** `precios_arriendo_barrios`  
**Split:** `train`  
**Fuente:** [Ficha del dataset en Hugging Face](https://huggingface.co/datasets/aptologia/precios-arriendo-colombia)



## 1. Preparar el entorno

Se instalan las bibliotecas necesarias para obtener el dataset y analizarlo con pandas.

In [27]:
%pip -q install -U datasets "pandas==2.2.3"

In [28]:
import pandas as pd
print("Versión de pandas:", pd.__version__)

Versión de pandas: 3.1.0


## 2. Importar bibliotecas y definir la fuente

Se declara el identificador, la configuración y el split. Se usará el dataset completo publicado en el split seleccionado; no se toma una muestra intencional.

In [29]:
from pathlib import Path
from datetime import datetime
import pandas as pd
from datasets import load_dataset
from IPython.display import display

DATASET_ID = "aptologia/precios-arriendo-colombia"
CONFIGURACION = "precios_arriendo_barrios"
SPLIT = "train"
URL_CSV = (
    "https://huggingface.co/datasets/"
    "aptologia/precios-arriendo-colombia/resolve/main/"
    "precios_arriendo_barrios.csv"
)

print("Dataset:", DATASET_ID)
print("Configuración:", CONFIGURACION)
print("Split:", SPLIT)

Dataset: aptologia/precios-arriendo-colombia
Configuración: precios_arriendo_barrios
Split: train


## 3. Obtener los datos desde Hugging Face

Primero se utiliza `load_dataset` con el repositorio, la configuración y el split. Si el cargador no logra resolver la configuración por un cambio de metadatos del repositorio, se intenta leer el CSV principal alojado en el mismo repositorio de Hugging Face. En ambos casos la fuente de los datos es el repositorio indicado.

In [30]:
try:
    dataset_hf = load_dataset(DATASET_ID, CONFIGURACION, split=SPLIT)
    df = dataset_hf.to_pandas()
    metodo_carga = "Hugging Face Datasets: load_dataset"
except Exception as error_carga:
    print("No fue posible cargar con load_dataset.")
    print("Detalle:", repr(error_carga))
    print("Se intentará el CSV principal del mismo repositorio de Hugging Face...")
    df = pd.read_csv(URL_CSV)
    metodo_carga = "CSV principal alojado en Hugging Face (fallback)"

if df.empty:
    raise ValueError("El DataFrame está vacío. Verifica la disponibilidad del dataset.")

print("Método de carga:", metodo_carga)
print("Dimensiones:", df.shape)
print("Primeros registros:")
display(df.head())

Método de carga: Hugging Face Datasets: load_dataset
Dimensiones: (1460, 10)
Primeros registros:


,fecha,ciudad_id,ciudad,barrio,avisos,p25,mediana,p75,con_parqueadero,con_telefono
0,2026-10-05,medellin,Medellín,El Poblado,17124,4400000,5700000,7700000,15907,6599
1,2026-10-05,medellin,Medellín,Poblado,7593,4100000,5650000,8000000,6509,841
2,2026-10-05,medellin,Medellín,Laureles,5780,3300000,4200000,5300000,4283,2169
3,2026-10-05,medellin,Medellín,Castropol,2730,4200000,5350000,7000000,2286,1547
4,2026-10-05,medellin,Medellín,El Tesoro,2471,5800000,7700000,11500000,2167,1278


## 4. Descripción básica del DataFrame

Cada fila representa un registro agregado por barrio y ciudad en la fotografía de datos; no representa un apartamento individual. Las columnas de precios `p25`, `mediana` y `p75` están expresadas como precios pedidos mensuales en pesos colombianos, según la ficha del dataset. Las variables `avisos`, `con_parqueadero` y `con_telefono` son conteos.

In [31]:
print("Cantidad de filas:", df.shape[0])
print("Cantidad de columnas:", df.shape[1])

print("\nNombres de columnas:")
print(df.columns.tolist())

print("\nTipos de datos:")
display(df.dtypes.rename("tipo_de_dato").to_frame())

print("\nValores nulos por columna:")
nulos = df.isna().sum()
display(nulos.rename("valores_nulos").to_frame())

print("\nInformación general:")
df.info()

Cantidad de filas: 1460
Cantidad de columnas: 10

Nombres de columnas:
['fecha', 'ciudad_id', 'ciudad', 'barrio', 'avisos', 'p25', 'mediana', 'p75', 'con_parqueadero', 'con_telefono']

Tipos de datos:


,tipo_de_dato
fecha,str
ciudad_id,str
ciudad,str
barrio,str
avisos,int64
p25,int64
mediana,int64
p75,int64
con_parqueadero,int64
con_telefono,int64



Valores nulos por columna:


,valores_nulos
fecha,0
ciudad_id,0
ciudad,0
barrio,0
avisos,0
p25,0
mediana,0
p75,0
con_parqueadero,0
con_telefono,0



Información general:
<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   fecha            1460 non-null   str  
 1   ciudad_id        1460 non-null   str  
 2   ciudad           1460 non-null   str  
 3   barrio           1460 non-null   str  
 4   avisos           1460 non-null   int64
 5   p25              1460 non-null   int64
 6   mediana          1460 non-null   int64
 7   p75              1460 non-null   int64
 8   con_parqueadero  1460 non-null   int64
 9   con_telefono     1460 non-null   int64
dtypes: int64(6), str(4)
memory usage: 167.4 KB


## 5. Reconocimiento de variables relevantes

Las variables que respaldan la necesidad propuesta son ciudad y barrio para comparar ubicaciones; `mediana` para describir el precio pedido típico; `p25` y `p75` para observar la distribución central de precios; y `avisos` para contextualizar cuántos avisos se consideraron en cada registro. `con_parqueadero` y `con_telefono` describen conteos de avisos con esas características.

In [32]:
variables_relevantes = [
    columna for columna in
    ["fecha", "ciudad", "barrio", "avisos", "p25", "mediana", "p75",
     "con_parqueadero", "con_telefono"]
    if columna in df.columns
]
print("Variables relevantes disponibles:", variables_relevantes)
display(df[variables_relevantes].head(10))

Variables relevantes disponibles: ['fecha', 'ciudad', 'barrio', 'avisos', 'p25', 'mediana', 'p75', 'con_parqueadero', 'con_telefono']


,fecha,ciudad,barrio,avisos,p25,mediana,p75,con_parqueadero,con_telefono
0,2026-10-05,Medellín,El Poblado,17124,4400000,5700000,7700000,15907,6599
1,2026-10-05,Medellín,Poblado,7593,4100000,5650000,8000000,6509,841
2,2026-10-05,Medellín,Laureles,5780,3300000,4200000,5300000,4283,2169
3,2026-10-05,Medellín,Castropol,2730,4200000,5350000,7000000,2286,1547
4,2026-10-05,Medellín,El Tesoro,2471,5800000,7700000,11500000,2167,1278
5,2026-10-05,Medellín,Santa Maria de los Angeles,2237,4100000,5600000,6800000,1930,922
6,2026-10-05,Medellín,Calasanz,2071,2200000,2600000,3200000,1175,742
7,2026-10-05,Medellín,Los Balsos,1870,4900000,6175000,9000000,1490,645
8,2026-10-05,Medellín,Las Palmas,1808,3900000,4300000,6200000,1503,659
9,2026-10-05,Medellín,Ciudad del Rio,1740,4000000,4500000,5412500,1372,900


## 6. Generar automáticamente el resumen `.txt`

El resumen se calcula a partir del mismo DataFrame cargado y revisado en este notebook. La carpeta se crea si no existe, el archivo se escribe en UTF-8 y se reemplaza cada vez que se ejecuta esta celda.

In [33]:
carpeta_resultados = Path("AE2") / "resultados"
carpeta_resultados.mkdir(parents=True, exist_ok=True)
ruta_resumen = carpeta_resultados / "resumen_dataframe.txt"

lineas = [
    "RESUMEN DEL DATAFRAME",
    "=" * 70,
    f"Identificador del dataset: {DATASET_ID}",
    f"Configuración: {CONFIGURACION}",
    f"Split: {SPLIT}",
    f"Método de carga: {metodo_carga}",
    "Muestra: no se seleccionó una muestra; se utilizó el split completo cargado.",
    "Unidad de análisis: registro agregado por ciudad y barrio en la fotografía disponible.",
    f"Fecha de generación del resumen: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "",
    f"Cantidad de filas: {df.shape[0]}",
    f"Cantidad de columnas: {df.shape[1]}",
    "",
    "NOMBRES Y TIPOS DE COLUMNAS",
    "-" * 70,
]
for nombre, tipo in df.dtypes.items():
    lineas.append(f"- {nombre}: {tipo}")

lineas.extend(["", "VALORES NULOS POR COLUMNA", "-" * 70])
for nombre, cantidad in df.isna().sum().items():
    lineas.append(f"- {nombre}: {int(cantidad)}")

lineas.extend([
    "",
    "PRIMEROS CINCO REGISTROS (REFERENCIA)",
    "-" * 70,
    df.head().to_string(index=False),
    "",
    "Nota: el resumen describe el DataFrame cargado. La ficha del dataset debe consultarse para conocer metodología, licencia y fecha de la fotografía.",
    "Fuente: https://huggingface.co/datasets/aptologia/precios-arriendo-colombia",
    "Fuente original declarada: Aptología, https://www.aptologia.com/datos-abiertos",
])

ruta_resumen.write_text("\n".join(lineas) + "\n", encoding="utf-8")
print("Resumen generado en:", ruta_resumen.resolve())
print("\nContenido del archivo:")
print(ruta_resumen.read_text(encoding="utf-8"))

Resumen generado en: /content/AE2/resultados/resumen_dataframe.txt

Contenido del archivo:
RESUMEN DEL DATAFRAME
Identificador del dataset: aptologia/precios-arriendo-colombia
Configuración: precios_arriendo_barrios
Split: train
Método de carga: Hugging Face Datasets: load_dataset
Muestra: no se seleccionó una muestra; se utilizó el split completo cargado.
Unidad de análisis: registro agregado por ciudad y barrio en la fotografía disponible.
Fecha de generación del resumen: 2026-10-22 20:08:10

Cantidad de filas: 1460
Cantidad de columnas: 10

NOMBRES Y TIPOS DE COLUMNAS
----------------------------------------------------------------------
- fecha: str
- ciudad_id: str
- ciudad: str
- barrio: str
- avisos: int64
- p25: int64
- mediana: int64
- p75: int64
- con_parqueadero: int64
- con_telefono: int64

VALORES NULOS POR COLUMNA
----------------------------------------------------------------------
- fecha: 0
- ciudad_id: 0
- ciudad: 0
- barrio: 0
- avisos: 0
- p25: 0
- mediana: 0
- p75

## 7. Verificación final

Por otro lado, Antes de entregar, confirmar que el archivo se creó y que contiene el identificador, la configuración, el split, las dimensiones, los tipos de datos y los valores nulos. Ejecuta esta celda para verificarlo.

In [34]:
assert ruta_resumen.exists(), "No se generó el archivo de resumen."
contenido_resumen = ruta_resumen.read_text(encoding="utf-8")
for texto_requerido in [
    DATASET_ID,
    f"Cantidad de filas: {df.shape[0]}",
    f"Cantidad de columnas: {df.shape[1]}",
    "VALORES NULOS POR COLUMNA",
]:
    assert texto_requerido in contenido_resumen, f"Falta en el resumen: {texto_requerido}"

print("Verificación correcta: el archivo existe y contiene los elementos requeridos.")

Verificación correcta: el archivo existe y contiene los elementos requeridos.


## 8. Descargar el resumen desde Google Colab (opcional)

Esta celda permite descargar el archivo generado para publicarlo en `AE2/resultados/` dentro del repositorio de GitHub.

In [35]:
from google.colab import files
files.download(str(ruta_resumen))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Referencias

- Aptología. *Precios de arriendo por barrio en Colombia*. Hugging Face Datasets. https://huggingface.co/datasets/aptologia/precios-arriendo-colombia
- Aptología. *Datos abiertos del arriendo en Colombia*. https://www.aptologia.com/datos-abiertos
- Hugging Face. *Load a dataset*. https://huggingface.co/docs/datasets/loading
- pandas. *Intro to data structures*. https://pandas.pydata.org/docs/user_guide/dsintro.html

## Nota sobre el alcance

Esta etapa es descriptiva y de reconocimiento de datos. Los valores representan precios pedidos publicados en avisos, no necesariamente precios finales pactados en contratos. El DataFrame está agregado por barrio y ciudad, por lo que no permite estimar el precio de un apartamento individual ni establecer relaciones causales.